In [0]:
# Databricks source notebook: AppLab Advanced ETL Pipeline
# Senior Data Engineer Technical Assessment
#
# Execution model:
#   1. Read and profile source CSV files.
#   2. Clean, validate and enrich sales records.
#   3. Fetch currency rates and convert valid sales to USD.
#   4. Archive rejected records and error logs as CSV before applying the threshold.
#   5. Export CSV files for import into the existing SQL Server tables through SSMS.
#
# Important:
#   - This assessment uses a small sample dataset. coalesce(1) is used only to
#     create one convenient CSV part file per output for the SSMS Import Wizard.
#   - For production-scale data, retain multiple output partitions and use a
#     managed database connectivity pattern rather than manual CSV transfer.
#   - The rejection threshold is evaluated on deduplicated source records.
#   - If the threshold is exceeded, rejected records and error logs are still
#     archived/exported, but FactSales is not exported as an accepted batch.
#
# Databricks paths below match the paths in the uploaded notebook. Change them
# only if the files are stored in a different Unity Catalog Volume.

# COMMAND ----------
# MAGIC %md
# MAGIC # AppLab – Advanced ETL Pipeline
# MAGIC **Senior Data Engineer Technical Assessment**
# MAGIC
# MAGIC This notebook ingests sales and product reference CSV files, applies data
# MAGIC quality rules, enriches sales with product and currency information, and
# MAGIC prepares outputs for SQL Server.

# COMMAND ----------
# MAGIC %md
# MAGIC ## 01. Imports and configuration

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql import types as T

import requests
from decimal import Decimal
from datetime import datetime, timezone

# Pipeline configuration
PIPELINE_NAME = "AppLab_Sales_ETL"
TARGET_CURRENCY = "USD"
REJECTION_THRESHOLD = 0.05

# External exchange-rate API (EUR is the base currency)
EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/EUR"
API_TIMEOUT_SECONDS = 15

# Run identifier makes archive/output folders unique across executions
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 02. Input and output paths

# COMMAND ----------
BASE_PATH = "/Volumes/workspace/default/source_data"

SALES_FILE_PATH = f"{BASE_PATH}/sales_data 2.csv"
PRODUCT_FILE_PATH = f"{BASE_PATH}/product_reference 2.csv"

# CSV archive paths are separate from the older Delta locations to avoid
# format conflicts from previous notebook runs.
REJECTED_ARCHIVE_PATH = f"{BASE_PATH}/rejected_records_csv/run_id={RUN_ID}"
ERROR_ARCHIVE_PATH = f"{BASE_PATH}/error_logs_csv/run_id={RUN_ID}"
CURRENCY_AUDIT_PATH = f"{BASE_PATH}/currency_audit/run_id={RUN_ID}"

SSMS_EXPORT_BASE_PATH = f"{BASE_PATH}/ssms_export/run_id={RUN_ID}"
FACT_SALES_EXPORT_PATH = f"{SSMS_EXPORT_BASE_PATH}/FactSales"
REJECTED_EXPORT_PATH = f"{SSMS_EXPORT_BASE_PATH}/RejectedRecords"
ERROR_LOG_EXPORT_PATH = f"{SSMS_EXPORT_BASE_PATH}/ErrorLogs"
CURRENCY_AUDIT_EXPORT_PATH = f"{SSMS_EXPORT_BASE_PATH}/CurrencyAudit"

# Confirm that the expected source files are available.
display(dbutils.fs.ls(BASE_PATH))

# COMMAND ----------
# MAGIC %md
# MAGIC ## 03. Custom schemas and data ingestion
# MAGIC
# MAGIC Explicit schemas avoid relying on schema inference. Decimal types are used
# MAGIC for monetary values to avoid binary floating-point precision issues.

# COMMAND ----------
sales_schema = T.StructType([
    T.StructField("OrderID", T.IntegerType(), True),
    T.StructField("ProductID", T.StringType(), True),
    T.StructField("SaleAmount", T.DecimalType(18, 2), True),
    T.StructField("OrderDate", T.StringType(), True),
    T.StructField("Region", T.StringType(), True),
    T.StructField("CustomerID", T.StringType(), True),
    T.StructField("Discount", T.DecimalType(10, 4), True),
    T.StructField("Currency", T.StringType(), True)
])

product_schema = T.StructType([
    T.StructField("ProductID", T.StringType(), True),
    T.StructField("ProductName", T.StringType(), True),
    T.StructField("Category", T.StringType(), True)
])

sales_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(sales_schema)
    .load(SALES_FILE_PATH)
)

product_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(product_schema)
    .load(PRODUCT_FILE_PATH)
)

print("Sales source schema:")
sales_raw_df.printSchema()

print("Product reference schema:")
product_raw_df.printSchema()

print("Sales source records:", sales_raw_df.count())
print("Product reference records:", product_raw_df.count())

# COMMAND ----------
# MAGIC %md
# MAGIC ## 04. Initial data profiling
# MAGIC
# MAGIC Profile the incoming data before applying cleaning rules. This establishes
# MAGIC a baseline for nulls, duplicates, date formats and currency distribution.

# COMMAND ----------
def display_null_profile(dataframe, dataset_name):
    """Display the null count for each column in a DataFrame."""
    null_profile = dataframe.select([
        F.count(
            F.when(F.col(column_name).isNull(), 1)
        ).alias(column_name)
        for column_name in dataframe.columns
    ])

    print(f"Null profile: {dataset_name}")
    display(null_profile)


display(sales_raw_df.limit(10))
display(product_raw_df.limit(10))

display_null_profile(sales_raw_df, "Sales")
display_null_profile(product_raw_df, "Product reference")

source_record_count = sales_raw_df.count()
exact_distinct_count = sales_raw_df.distinct().count()
exact_duplicate_count = source_record_count - exact_distinct_count

print("Source records:", source_record_count)
print("Distinct complete rows:", exact_distinct_count)
print("Exact duplicate rows:", exact_duplicate_count)

print("OrderID frequency:")
display(
    sales_raw_df
    .groupBy("OrderID")
    .count()
    .filter(F.col("count") > 1)
)

print("Currency distribution:")
display(sales_raw_df.groupBy("Currency").count())

print("Distinct source date values:")
display(
    sales_raw_df
    .select("OrderDate")
    .distinct()
    .orderBy("OrderDate")
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 05. Data cleaning
# MAGIC
# MAGIC Standardize text fields, normalize blank strings to NULL, parse supported
# MAGIC date formats, and apply the documented default discount of zero when the
# MAGIC source discount is missing. Missing critical fields are retained for
# MAGIC validation and rejection rather than silently discarded.

# COMMAND ----------
sales_clean_df = (
    sales_raw_df
    .withColumn("ProductID", F.trim(F.col("ProductID")))
    .withColumn("CustomerID", F.trim(F.col("CustomerID")))
    .withColumn("Region", F.trim(F.col("Region")))
    .withColumn("Currency", F.upper(F.trim(F.col("Currency"))))
    .withColumn("OrderDate", F.trim(F.col("OrderDate")))
    .withColumn(
        "ProductID",
        F.when(F.col("ProductID") == "", F.lit(None)).otherwise(F.col("ProductID"))
    )
    .withColumn(
        "CustomerID",
        F.when(F.col("CustomerID") == "", F.lit(None)).otherwise(F.col("CustomerID"))
    )
    .withColumn(
        "Region",
        F.when(F.col("Region") == "", F.lit(None)).otherwise(F.col("Region"))
    )
    .withColumn(
        "OrderDate",
        F.when(F.col("OrderDate") == "", F.lit(None)).otherwise(F.col("OrderDate"))
    )
    .withColumn(
        "OrderDateParsed",
        F.coalesce(
            F.to_date(F.expr("try_to_timestamp(OrderDate, 'dd/MM/yyyy')")),
            F.to_date(F.expr("try_to_timestamp(OrderDate, 'dd-MM-yyyy')")),
            F.to_date(F.expr("try_to_timestamp(OrderDate, 'yyyy-MM-dd')"))
        )
    )
    # Business assumption: a missing discount means no discount was applied.
    .withColumn("Discount", F.coalesce(F.col("Discount"), F.lit(Decimal("0.0000"))))
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 06. Data validation
# MAGIC
# MAGIC Apply business rules and attach all applicable rejection reasons to each
# MAGIC invalid record. Product existence is checked here; the descriptive product
# MAGIC lookup is performed in Section 08.

# COMMAND ----------
# Product reference keys are used to identify missing product IDs.
product_keys_df = (
    product_raw_df
    .select("ProductID")
    .dropDuplicates()
    .withColumn("ProductMatch", F.lit(1))
)

sales_validation_df = (
    sales_clean_df
    .join(product_keys_df, on="ProductID", how="left")
    .withColumn(
        "RejectionReason",
        F.concat_ws(
            "|",
            F.when(F.col("OrderID").isNull(), F.lit("MISSING_ORDER_ID")),
            F.when(F.col("SaleAmount").isNull(), F.lit("MISSING_SALE_AMOUNT")),
            F.when(F.col("SaleAmount") < 0, F.lit("NEGATIVE_SALE_AMOUNT")),
            F.when(F.col("OrderDate").isNull(), F.lit("MISSING_ORDER_DATE")),
            F.when(
                F.col("OrderDate").isNotNull() & F.col("OrderDateParsed").isNull(),
                F.lit("INVALID_ORDER_DATE")
            ),
            F.when(F.col("CustomerID").isNull(), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("ProductID").isNull(), F.lit("MISSING_PRODUCT_ID")),
            F.when(F.col("ProductMatch").isNull(), F.lit("PRODUCT_NOT_FOUND")),
            F.when(
                ~F.col("Currency").isin("USD", "EUR", "GBP"),
                F.lit("UNSUPPORTED_CURRENCY")
            ),
            F.when(
                (F.col("Discount") < 0) | (F.col("Discount") > 1),
                F.lit("INVALID_DISCOUNT")
            )
        )
    )
)

valid_sales_df = sales_validation_df.filter(F.col("RejectionReason") == "")
rejected_sales_df = sales_validation_df.filter(F.col("RejectionReason") != "")

print("Records after cleaning:", sales_validation_df.count())
print("Valid records:", valid_sales_df.count())
print("Rejected records:", rejected_sales_df.count())

display(
    rejected_sales_df.select(
        "OrderID", "CustomerID", "ProductID", "SaleAmount",
        "Currency", "OrderDate", "RejectionReason"
    )
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 07. Duplicate removal
# MAGIC
# MAGIC Remove exact duplicate transaction rows. We do not deduplicate by OrderID
# MAGIC alone because a production order can legitimately contain multiple lines.
# MAGIC The source sample contains an exact duplicate of a complete row.

# COMMAND ----------
sales_dedup_df = sales_validation_df.dropDuplicates()

# Rebuild valid/rejected datasets from the deduplicated data so duplicate rows
# are not counted more than once in the quality metrics or downstream outputs.
valid_sales_df = sales_dedup_df.filter(F.col("RejectionReason") == "")
rejected_sales_df = sales_dedup_df.filter(F.col("RejectionReason") != "")

deduplicated_record_count = sales_dedup_df.count()
duplicates_removed = source_record_count - deduplicated_record_count

print("Source records:", source_record_count)
print("Records after exact deduplication:", deduplicated_record_count)
print("Exact duplicates removed:", duplicates_removed)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 08. Product lookup and enrichment
# MAGIC
# MAGIC Check that the product reference contains one row per ProductID before
# MAGIC joining. This prevents a many-to-many join from multiplying sales rows.

# COMMAND ----------
duplicate_product_keys_df = (
    product_raw_df
    .groupBy("ProductID")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_product_key_count = duplicate_product_keys_df.count()

if duplicate_product_key_count > 0:
    display(duplicate_product_keys_df)
    raise ValueError(
        "Product reference contains duplicate ProductID values. "
        "Resolve the reference data before performing the lookup."
    )

product_lookup_df = product_raw_df.select(
    "ProductID",
    "ProductName",
    "Category"
)

sales_enriched_df = valid_sales_df.join(
    product_lookup_df,
    on="ProductID",
    how="inner"
)

if sales_enriched_df.count() != valid_sales_df.count():
    raise ValueError(
        "Product lookup changed the valid sales record count. "
        "Review product reference completeness and uniqueness."
    )

display(
    sales_enriched_df.select(
        "OrderID", "CustomerID", "ProductID", "ProductName", "Category",
        "SaleAmount", "Currency", "OrderDateParsed", "Discount", "Region"
    )
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 09. Currency API integration
# MAGIC
# MAGIC The API uses EUR as its base currency. EUR-to-USD is read directly from
# MAGIC the response. GBP-to-USD is calculated as EUR-to-USD divided by EUR-to-GBP.
# MAGIC If the API call or response validation fails, documented fallback rates
# MAGIC are used and the rate source is recorded.

# COMMAND ----------
FALLBACK_EXCHANGE_RATES = {
    "USD": Decimal("1.000000"),
    "EUR": Decimal("1.100000"),
    "GBP": Decimal("1.270000")
}

def fetch_exchange_rates():
    """Return currency-to-USD rates and the source used (API or FALLBACK)."""
    try:
        response = requests.get(
            EXCHANGE_API_URL,
            timeout=API_TIMEOUT_SECONDS
        )
        response.raise_for_status()

        api_data = response.json()
        api_rates = api_data["rates"]

        eur_to_usd = Decimal(str(api_rates["USD"]))
        eur_to_gbp = Decimal(str(api_rates["GBP"]))

        if eur_to_usd <= 0 or eur_to_gbp <= 0:
            raise ValueError("API returned a non-positive exchange rate.")

        gbp_to_usd = eur_to_usd / eur_to_gbp

        rates = {
            "USD": Decimal("1.000000"),
            "EUR": eur_to_usd.quantize(Decimal("0.000001")),
            "GBP": gbp_to_usd.quantize(Decimal("0.000001"))
        }

        print("Exchange rates retrieved successfully.")
        return rates, "API"

    except Exception as api_error:
        print(f"Currency API failed: {api_error}")
        print("Using configured fallback exchange rates.")
        return FALLBACK_EXCHANGE_RATES, "FALLBACK"


exchange_rates, rate_source = fetch_exchange_rates()

print("Rate source:", rate_source)
print("Rates to USD:", exchange_rates)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 10. Currency conversion and conversion audit
# MAGIC
# MAGIC Join the currency lookup to valid sales and calculate SaleAmountUSD.
# MAGIC Record the rate, source, timestamp and transaction identifiers for audit.

# COMMAND ----------
exchange_rate_schema = T.StructType([
    T.StructField("Currency", T.StringType(), False),
    T.StructField("ExchangeRateToUSD", T.DecimalType(18, 6), False)
])

exchange_rate_df = spark.createDataFrame(
    [
        (currency, rate)
        for currency, rate in exchange_rates.items()
    ],
    schema=exchange_rate_schema
)

sales_with_rates_df = sales_enriched_df.join(
    exchange_rate_df,
    on="Currency",
    how="left"
)

missing_rate_count = (
    sales_with_rates_df
    .filter(F.col("ExchangeRateToUSD").isNull())
    .count()
)

if missing_rate_count > 0:
    raise ValueError(
        f"{missing_rate_count} valid sales records do not have an exchange rate."
    )

sales_converted_df = (
    sales_with_rates_df
    .withColumn(
        "SaleAmountUSD",
        F.round(
            F.col("SaleAmount") * F.col("ExchangeRateToUSD"),
            2
        ).cast(T.DecimalType(18, 2))
    )
)

currency_audit_df = (
    sales_converted_df
    .select(
        "OrderID",
        "CustomerID",
        "ProductID",
        "Currency",
        "SaleAmount",
        "ExchangeRateToUSD",
        "SaleAmountUSD"
    )
    .withColumn("RateSource", F.lit(rate_source))
    .withColumn("ConversionTimestamp", F.current_timestamp())
)

display(
    sales_converted_df.select(
        "OrderID", "Currency", "SaleAmount",
        "ExchangeRateToUSD", "SaleAmountUSD"
    )
)

display(currency_audit_df)

# Persist the conversion audit independently of the FactSales acceptance gate.
(
    currency_audit_df.write
    .format("delta")
    .mode("overwrite")
    .save(CURRENCY_AUDIT_PATH)
)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 11. Error logging and rejected-record archival
# MAGIC
# MAGIC Preserve rejected records and create one error-log row per rejection reason.
# MAGIC Archive these datasets before evaluating the rejection threshold so that
# MAGIC audit evidence is retained even when the batch fails quality acceptance.

# COMMAND ----------
rejected_records_df = (
    rejected_sales_df
    .select(
        "OrderID",
        "CustomerID",
        "ProductID",
        "SaleAmount",
        "Currency",
        "OrderDate",
        "RejectionReason"
    )
    .withColumn("ErrorTimestamp", F.current_timestamp())
    .withColumn("PipelineName", F.lit(PIPELINE_NAME))
)

error_logs_df = (
    rejected_records_df
    .select(
        "OrderID",
        "CustomerID",
        "ProductID",
        F.explode(
            F.split(F.col("RejectionReason"), r"\|")
        ).alias("ErrorType"),
        "ErrorTimestamp",
        "PipelineName"
    )
)

# Archive rejected records and error logs as CSV.
# These are new CSV-specific paths, so they cannot conflict with the old
# Delta folders created during earlier troubleshooting attempts.
(
    rejected_records_df.coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", True)
    .option("emptyValue", "")
    .save(REJECTED_ARCHIVE_PATH)
)

(
    error_logs_df.coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", True)
    .option("emptyValue", "")
    .save(ERROR_ARCHIVE_PATH)
)

print("Archived rejected records:", rejected_records_df.count())
print("Rejected records archive:", REJECTED_ARCHIVE_PATH)
print("Archived error log entries:", error_logs_df.count())
print("Error logs archive:", ERROR_ARCHIVE_PATH)

display(rejected_records_df)
display(error_logs_df)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 12. Rejection threshold validation
# MAGIC
# MAGIC The rejection rate is calculated using the deduplicated input record count.
# MAGIC The batch is accepted only when the rejection rate is at or below 5%.
# MAGIC Rejected records and error logs have already been archived in Section 11.

# COMMAND ----------
total_processed_records = deduplicated_record_count
rejected_record_count = rejected_records_df.count()

rejection_rate = (
    rejected_record_count / total_processed_records
    if total_processed_records > 0
    else 0.0
)

threshold_passed = rejection_rate <= REJECTION_THRESHOLD

print("Deduplicated input records:", total_processed_records)
print("Rejected records:", rejected_record_count)
print(f"Rejection rate: {rejection_rate:.2%}")
print(f"Allowed threshold: {REJECTION_THRESHOLD:.2%}")
print("Threshold passed:", threshold_passed)

if not threshold_passed:
    print(
        "BATCH REJECTED: FactSales will not be exported. "
        "Rejected records and error logs have been archived."
    )

# COMMAND ----------
# MAGIC %md
# MAGIC ## 13. Final transformations
# MAGIC
# MAGIC Prepare the FactSales dataset only when the rejection threshold passes.
# MAGIC Column names and data types are aligned with the existing SQL Server table.

# COMMAND ----------
final_sales_df = None

if threshold_passed:
    final_sales_df = (
        sales_converted_df
        .select(
            F.col("OrderID").cast(T.IntegerType()).alias("OrderID"),
            F.col("CustomerID").cast(T.StringType()).alias("CustomerID"),
            F.col("ProductID").cast(T.StringType()).alias("ProductID"),
            F.col("ProductName").cast(T.StringType()).alias("ProductName"),
            F.col("Category").cast(T.StringType()).alias("Category"),
            F.col("SaleAmount").cast(T.DecimalType(18, 2)).alias("SaleAmount"),
            F.col("Currency").cast(T.StringType()).alias("Currency"),
            F.col("ExchangeRateToUSD").cast(T.DecimalType(18, 6)).alias("ExchangeRateToUSD"),
            F.col("SaleAmountUSD").cast(T.DecimalType(18, 2)).alias("SaleAmountUSD"),
            F.col("OrderDateParsed").cast(T.DateType()).alias("OrderDate"),
            F.col("Discount").cast(T.DecimalType(10, 4)).alias("Discount"),
            F.col("Region").cast(T.StringType()).alias("Region")
        )
    )

    print("Final FactSales schema:")
    final_sales_df.printSchema()
    print("Final FactSales records:", final_sales_df.count())
    display(final_sales_df)
else:
    print("Final FactSales preparation skipped because the batch failed the threshold.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 14. Prepare CSV exports for SQL Server / SSMS
# MAGIC
# MAGIC This assessment uses SSMS Import and Export Wizard rather than a direct JDBC
# MAGIC connection. The CSV files are written to separate folders in the Volume.
# MAGIC ErrorLogs excludes ErrorID because SQL Server generates it using IDENTITY.
# MAGIC FactSales is exported only for a batch that passes the rejection threshold.

# COMMAND ----------
def export_csv_for_ssms(dataframe, output_path, dataset_name):
    """
    Export a small assessment dataset as a single-header CSV part file.
    coalesce(1) is for convenient manual import into SSMS, not large-scale use.
    """
    (
        dataframe.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .option("emptyValue", "")
        .csv(output_path)
    )

    print(f"{dataset_name} exported to: {output_path}")
    display(dbutils.fs.ls(output_path))


# Rejected records and error logs are exported even when the batch is rejected.
export_csv_for_ssms(
    rejected_records_df,
    REJECTED_EXPORT_PATH,
    "RejectedRecords"
)

export_csv_for_ssms(
    error_logs_df,
    ERROR_LOG_EXPORT_PATH,
    "ErrorLogs"
)

export_csv_for_ssms(
    currency_audit_df,
    CURRENCY_AUDIT_EXPORT_PATH,
    "CurrencyAudit"
)

# Only accepted batches are exported for insertion into dbo.FactSales.
if threshold_passed:
    export_csv_for_ssms(
        final_sales_df,
        FACT_SALES_EXPORT_PATH,
        "FactSales"
)
else:
    print("FactSales CSV not generated: rejection threshold was exceeded.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 15. Final validation and reconciliation
# MAGIC
# MAGIC Reconcile the original input, exact duplicates removed, valid records and
# MAGIC rejected records. SQL Server table counts can be checked after the CSV files
# MAGIC are imported through SSMS.

# COMMAND ----------
valid_record_count = valid_sales_df.count()
rejected_record_count = rejected_records_df.count()

print("========== PIPELINE RECONCILIATION ==========")
print("Original source records:", source_record_count)
print("Exact duplicates removed:", duplicates_removed)
print("Deduplicated records:", deduplicated_record_count)
print("Valid records:", valid_record_count)
print("Rejected records:", rejected_record_count)
print("Valid + rejected:", valid_record_count + rejected_record_count)
print(f"Rejection rate: {rejection_rate:.2%}")
print(f"Threshold: {REJECTION_THRESHOLD:.2%}")
print("Threshold passed:", threshold_passed)

assert (
    deduplicated_record_count == valid_record_count + rejected_record_count
), "Reconciliation failed: valid + rejected does not equal deduplicated input."

print("Reconciliation passed.")

# Fail the batch only after rejected data, error logs and audit outputs are saved.
# This prevents rejected batches from being treated as accepted FactSales loads.
if not threshold_passed:
    raise RuntimeError(
        f"ETL batch failed quality acceptance: rejection rate "
        f"{rejection_rate:.2%} exceeds the allowed "
        f"{REJECTION_THRESHOLD:.2%}. Rejected records and error logs "
        f"were archived and exported. FactSales was not exported."
    )

print("ETL batch passed quality acceptance.")
print("FactSales CSV is ready for import into SQL Server using SSMS.")


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/source_data/currency_audit/,currency_audit/,0,1794574960136
dbfs:/Volumes/workspace/default/source_data/error_logs/,error_logs/,0,1794574960136
dbfs:/Volumes/workspace/default/source_data/error_logs_csv/,error_logs_csv/,0,1794574960136
dbfs:/Volumes/workspace/default/source_data/product_reference 2.csv,product_reference 2.csv,351,1794297416000
dbfs:/Volumes/workspace/default/source_data/rejected_records/,rejected_records/,0,1794574960136
dbfs:/Volumes/workspace/default/source_data/rejected_records_csv/,rejected_records_csv/,0,1794574960136
dbfs:/Volumes/workspace/default/source_data/sales_data 2.csv,sales_data 2.csv,938,1794297416000
dbfs:/Volumes/workspace/default/source_data/ssms_export/,ssms_export/,0,1794574960136


Sales source schema:
root
 |-- OrderID: integer (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- SaleAmount: decimal(18,2) (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Discount: decimal(10,4) (nullable = true)
 |-- Currency: string (nullable = true)

Product reference schema:
root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)

Sales source records: 20
Product reference records: 10


OrderID,ProductID,SaleAmount,OrderDate,Region,CustomerID,Discount,Currency
1001,P50,299.99,01/05/2023,East,C100,0.1000,USD
1002,P72,null,01/05/2023,West,C101,null,EUR
1003,P50,-10.00,01-06-2023,East,C100,0.0500,GBP
1001,P50,299.99,01/05/2023,East,C100,0.1000,USD
1004,P99,150.00,null,South,C102,0.2000,USD
1005,PX1,89.50,01/07/2023,North,null,0.0000,USD
1006,P72,200.00,2023-13-01,West,C101,0.1500,EUR
1007,P12,120.00,01/05/2023,East,C105,0.1000,GBP
1008,P88,300.00,02/05/2023,North,C106,0.0000,USD
1009,P77,0.00,03/05/2023,South,C107,null,USD


ProductID,ProductName,Category
P50,Wireless Mouse,Electronics
P72,Laptop Backpack,Accessories
P99,USB Hub,Electronics
P12,Notebook Stationery,Office Supplies
P88,Monitor Stand,Office Supplies
P77,Portable Speaker,Electronics
P45,Smartphone Charger,Electronics
P11,Pen Set,Office Supplies
P66,Wireless Keyboard,Electronics
P23,Laptop Stand,Accessories


Null profile: Sales


OrderID,ProductID,SaleAmount,OrderDate,Region,CustomerID,Discount,Currency
0,0,1,1,0,1,3,0


... (22 outputs left out)

CurrencyAudit exported to: /Volumes/workspace/default/source_data/ssms_export/run_id=20261113T130227Z/CurrencyAudit


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T130227Z/CurrencyAudit/_SUCCESS,_SUCCESS,0,1794575015000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T130227Z/CurrencyAudit/_committed_2313370951013663374,_committed_2313370951013663374,113,1794575014000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T130227Z/CurrencyAudit/_started_2313370951013663374,_started_2313370951013663374,0,1794575014000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T130227Z/CurrencyAudit/part-00000-tid-2313370951013663374-ff689e58-b52d-4b5b-9047-a69cf7732ed4-272-1-c000.csv,part-00000-tid-2313370951013663374-ff689e58-b52d-4b5b-9047-a69cf7732ed4-272-1-c000.csv,1014,1794575014000


FactSales CSV not generated: rejection threshold was exceeded.
========== PIPELINE RECONCILIATION ==========
Original source records: 20
Exact duplicates removed: 1
Deduplicated records: 19
Valid records: 13
Rejected records: 6
Valid + rejected: 19
Rejection rate: 31.58%
Threshold: 5.00%
Threshold passed: False
Reconciliation passed.


---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-5794518339325838>, line 729
    726 # Fail the batch only after rejected data, error logs and audit outputs are saved.
    727 # This prevents rejected batches from being treated as accepted FactSales loads.
    728 if not threshold_passed:
--> 729     raise RuntimeError(
    730         f"ETL batch failed quality acceptance: rejection rate "
    731         f"{rejection_rate:.2%} exceeds the allowed "
    732         f"{REJECTION_THRESHOLD:.2%}. Rejected records and error logs "
    733         f"were archived and exported. FactSales was not exported."
    734     )
    736 print("ETL batch passed quality acceptance.")
    737 print("FactSales CSV is ready for import into SQL Server using SSMS.")

RuntimeError: ETL batch failed quality acceptance: rejection rate 31.58% exceeds the allowed 5.00%. Rejected records and error

In [0]:
display(dbutils.fs.ls(REJECTED_EXPORT_PATH))
display(dbutils.fs.ls(ERROR_LOG_EXPORT_PATH))
display(dbutils.fs.ls(CURRENCY_AUDIT_EXPORT_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/RejectedRecords/_SUCCESS,_SUCCESS,0,1794564882000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/RejectedRecords/_committed_3936820251963270441,_committed_3936820251963270441,113,1794564881000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/RejectedRecords/_started_3936820251963270441,_started_3936820251963270441,0,1794564881000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/RejectedRecords/part-00000-tid-3936820251963270441-452a7a94-6a9b-4dca-bee8-24be4152d800-700-1-c000.csv,part-00000-tid-3936820251963270441-452a7a94-6a9b-4dca-bee8-24be4152d800-700-1-c000.csv,688,1794564881000


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/ErrorLogs/_SUCCESS,_SUCCESS,0,1794564884000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/ErrorLogs/_committed_2016873097964808030,_committed_2016873097964808030,113,1794564884000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/ErrorLogs/_started_2016873097964808030,_started_2016873097964808030,0,1794564884000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/ErrorLogs/part-00000-tid-2016873097964808030-c47c3d4e-ccdc-4b9e-8870-c599e42de451-708-1-c000.csv,part-00000-tid-2016873097964808030-c47c3d4e-ccdc-4b9e-8870-c599e42de451-708-1-c000.csv,589,1794564884000


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/CurrencyAudit/_SUCCESS,_SUCCESS,0,1794564887000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/CurrencyAudit/_committed_9094260560331624471,_committed_9094260560331624471,113,1794564886000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/CurrencyAudit/_started_9094260560331624471,_started_9094260560331624471,0,1794564886000
dbfs:/Volumes/workspace/default/source_data/ssms_export/run_id=20261113T101412Z/CurrencyAudit/part-00000-tid-9094260560331624471-d8f59f3c-85b3-4a91-a9af-cf961c5fdbeb-720-1-c000.csv,part-00000-tid-9094260560331624471-d8f59f3c-85b3-4a91-a9af-cf961c5fdbeb-720-1-c000.csv,1014,1794564886000
